# 04｜指定 ViT Run 的样本外因子评价与回测

请使用 `ViT Factor Eval (Alphalens/Pyfolio)` 内核。三种口径都先在完整 T 日预测截面固定五分位，RankIC 使用完整价格有效截面。默认采用口径 2：T→T+k，不使用未来的 `can_enter_t1` 状态过滤；口径 1 和口径 3 可在参数单元显式选择。每次评价还会自动生成完整因子覆盖池的全市场等权基准及相对绩效指标。


## 步骤 1：选择实验 run 与回测口径

这里的 `FOLDER_NAME` 必须与 Notebook 03 完成训练的 run 完全一致。`BASELINE_FOLDER` 表示另一个模型 run；它与每次评价自动生成的全市场等权基准不是一回事。比较前需要用相同 `BACKTEST_MODE` 分别刷新两个 run 的评价。


In [ ]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()
VIT_DIR = CURRENT_DIR if (CURRENT_DIR / 'vit_stage08_factor_evaluation.py').exists() else CURRENT_DIR / 'ViT'
DATA_ROOT = VIT_DIR / 'data'
if str(VIT_DIR) not in sys.path:
    sys.path.insert(0, str(VIT_DIR))

BASELINE_FOLDER = 'date_batch'
FOLDER_NAME = 'sigmoid_rank_penalty_turnover'  # 只需输入 predictions 下的文件夹名
BACKTEST_MODE = 'mode2_t_no_filter'
# BACKTEST_MODE 全部可选值：
# 'mode1_t_with_t1_filter'：T→T+k；五分位固定后，仅组合收益和换手使用 can_enter_t1
# 'mode2_t_no_filter'：T→T+k；不使用未来状态过滤（默认）
# 'mode3_t1_no_filter'：T+1→T+1+k；不使用未来状态过滤（1/5/10D 终点为 T+2/T+6/T+11）

print({
    'ViT目录': str(VIT_DIR),
    '待评估run': FOLDER_NAME,
    '对比模型run': BASELINE_FOLDER,
    '回测口径': BACKTEST_MODE,
    '收益周期': (1, 5, 10),
    '自动市场基准': '完整因子覆盖池全市场等权',
})


## 步骤 2：执行所选 run 的 Alphalens / Pyfolio 评价

该内核无需安装 PyTorch。评价会替换所选 run 自己的 `factor_evaluation/` 与实验报告，不修改模型、预测或训练产物。输出包括评价 Parquet、累计收益与最大回撤图、Alphalens tear sheet，以及 `relative_performance.parquet`。


In [ ]:
from vit_stage08_factor_evaluation import run_oos_backtest

final_audit = run_oos_backtest(
    run_name=FOLDER_NAME,
    data_root=DATA_ROOT,
    backtest_mode=BACKTEST_MODE,
)


## 步骤 3：查看所选 run 的最终动态审计


In [ ]:
from IPython.display import display

display(final_audit)


## 步骤 4：与固定 batch 模型 run 对比

这里只读取两个 run 已生成的评价产物。代码会校验双方回测模式、价格起点、执行策略和全市场等权基准定义一致；旧 schema 或不同口径会明确要求重新运行步骤 2。


In [ ]:
import json
import pandas as pd

run_names = [BASELINE_FOLDER]
if FOLDER_NAME != BASELINE_FOLDER:
    run_names.append(FOLDER_NAME)

def validate_evaluation_policy(run_name):
    path = DATA_ROOT / "quality" / "runs" / run_name / "factor_evaluation" / "factor_evaluation_manifest.json"
    if not path.exists():
        raise FileNotFoundError(f"缺少评估 manifest：{path}")
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if int(manifest.get("schema_version", 0)) < 2 or "backtest_mode" not in manifest:
        raise RuntimeError(f"{run_name} 是旧评价产物；请用当前 Notebook 04 重新执行步骤 2。")
    if manifest["backtest_mode"] != BACKTEST_MODE:
        raise RuntimeError(
            f"{run_name} 的回测口径为 {manifest['backtest_mode']}，当前为 {BACKTEST_MODE}；"
            "请统一参数后重新评价。"
        )
    if manifest.get("benchmark_column") != "Equal-weight market":
        raise RuntimeError(f"{run_name} 缺少当前全市场等权基准定义；请重新评价。")
    return manifest

manifests = {name: validate_evaluation_policy(name) for name in run_names}
compatibility_keys = [
    "dataset_signature", "universe_signature", "grouping_policy",
    "periods", "quantiles", "backtest_mode", "price_start",
    "price_start_offset", "forward_return_formula", "portfolio_scope",
    "execution_mask", "benchmark_column", "benchmark_scope",
    "benchmark_weighting", "benchmark_execution_mask",
    "annual_excess_method", "annual_tracking_error_method",
    "information_ratio_method", "annualization_days",
    "long_short_gross_exposure",
]
for key in compatibility_keys:
    values = {name: manifests[name].get(key) for name in run_names}
    reference = values[run_names[0]]
    if any(value != reference for value in values.values()):
        raise RuntimeError(f"两个 run 的 {key} 不一致：{values}")

def read_run_table(run_name, file_name):
    path = DATA_ROOT / "quality" / "runs" / run_name / "factor_evaluation" / file_name
    if not path.exists():
        raise FileNotFoundError(f"缺少评估产物：{path}")
    table = pd.read_parquet(path).copy()
    table.insert(0, "run_name", run_name)
    return table

def compare_table(file_name):
    return pd.concat([read_run_table(name, file_name) for name in run_names], ignore_index=True)

print("RankIC 对比")
display(compare_table("ic_summary.parquet"))
print("Q5-Q1 对比")
display(compare_table("spread_summary.parquet"))
print("Pyfolio 绝对绩效对比（含全市场等权）")
display(compare_table("pyfolio_performance.parquet"))
print("相对全市场等权绩效对比")
display(compare_table("relative_performance.parquet"))
print({"本次实验报告": str(DATA_ROOT / "quality" / "runs" / FOLDER_NAME / "EXPERIMENT_REPORT.md")})
